🌐 [English](getting_started.ipynb) | **한국어** | [日本語](getting_started.ja.ipynb) | [中文](getting_started.zh.ipynb)

# DEEPX Tutorial 01 - DEEPX SDK 설치 방법
이 첫 번째 튜토리얼은 DEEPX SDK를 설치하고 설치가 정상적으로 끝났는지 확인하는 방법을 설명합니다. 환경을 준비하고, SDK를 설치하고, DEEPX NPU 장치(DX-M1, DX-M1M, DX-H1 Quattro)가 시스템에서 올바르게 인식되는지 확인하는 과정을 배웁니다.

## 학습 목표

이 튜토리얼을 마치면 DX-All Suite를 성공적으로 설치하고 DEEPX NPU 장치에서 기본 흐름을 실행할 수 있습니다.

## 사전 요구 사항

**참고:** 아래 요구 사항은 이 튜토리얼을 위한 것이며 **DEEPX 제품을 사용하기 위한 필수 조건은 아닙니다.**
- OS: Linux (Ubuntu 20.04/22.04/24.04/26.04, Debian 12/13)
- RAM: 8G (DX-Compiler 사용 시 16G)
- 저장 공간: 최소 40 GB
- DEEPX NPU: DX-M1, DX-M1M, DX-H1 Quattro
- CPU: x86_64에서는 DX-Compiler + DX-Runtime, aarch64에서는 DX-Runtime만 지원

## DXNN® - DEEPX NPU SDK 소개 (DX-AS: DX-All Suite)

DX-AS(DX-All Suite)는 DEEPX 장치로 AI 모델을 컴파일하고 추론할 수 있게 해 주는 프레임워크와 도구의 통합 환경입니다. 개별 도구를 하나씩 설치해 통합 환경을 구성할 수도 있지만, DX-AS는 개별 도구의 버전을 서로 맞춰 최적의 호환성을 유지합니다.

![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/resources/dxnn_sdk_illustration.png)

DEEPX SDK는 크게 두 부분으로 나뉩니다.

첫 번째는 AI 모델 컴파일 환경으로, AI 모델을 DEEPX NPU에서 효율적으로 실행할 수 있는 최적화된 형식으로 변환합니다.

두 번째는 AI 모델 런타임 환경으로, 컴파일된 AI 모델을 실제 DEEPX NPU 하드웨어에서 실행해 결과를 생성합니다.

DX-All Suite를 사용하면 두 구성 요소를 따로 관리하지 않고도 한 번에 설정할 수 있습니다.

![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/img/dx-as.png)


![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/img/DXNN-SDK-Simple-Architecture.png)



이해를 돕기 위한 DX-SDK 소개 YouTube 영상 두 편이 있습니다.
- [Youtube - DEEPX SDK Introduction](https://www.youtube.com/watch?v=Js6Soex0WI4) | 다운로드: [EN](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_ENG.mp4) · [中文](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_CH.mp4) · [한국어](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_KO.mp4)

<img src="assets/youtube-dx-sdk.png" style="max-width: 1000px;">

## 1. DX-All Suite 다운로드

### 1.1 설치 구성

튜토리얼은 다음 순서로 DX-All Suite를 찾습니다.

1. 환경 변수 `DX_ALL_SUITE_DIR`,
2. `dx-tutorials/config.json` (이 절에서 생성되며 git으로 추적되지 않습니다),
3. `~/dx-all-suite`, `dx-tutorials` 옆의 `dx-all-suite` 폴더 등 일반적인 위치 자동 탐지,
4. 기본 위치 `~/dx-all-suite`.

다음 셀은 찾은 위치와 그 값의 출처를 출력합니다. SDK가 아직 설치되지 않았다면 기본 위치가 `MISSING`으로 표시됩니다. 클론 단계 전에는 정상적인 상태입니다.

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial, save_tutorial_config

# Modules used by cells later in this notebook.
import shlex, shutil, subprocess
from IPython.display import Markdown, display

paths = setup_tutorial()


#### 1.1.1 설치 디렉터리

위에 출력된 위치를 그대로 사용하려면 다음 셀을 수정하지 마세요. DX-All Suite를 다른 곳에 설치하려는 경우(또는 이미 설치한 경우)에만 `DX_ALL_SUITE_DIR`을 수정합니다. 셀을 실행하면 선택한 값이 `dx-tutorials/config.json`에 저장되어 다른 모든 튜토리얼이 같은 위치를 사용합니다.

터미널에서 언제든지 설정할 수도 있습니다(`dx-tutorials` 디렉터리에서 실행).

```bash
python tutorial_paths.py --set ~/my/dx-all-suite
```

In [ ]:
####################################################################
# Keep the detected location, or edit this value to choose another one.
# Example: DX_ALL_SUITE_DIR = Path("~/my/dx-all-suite")
####################################################################
DX_ALL_SUITE_DIR = paths.dx_all_suite_dir
####################################################################

save_tutorial_config(dx_all_suite_dir=DX_ALL_SUITE_DIR)
paths = setup_tutorial()


#### 1.1.2 Git 브랜치

다른 SDK 버전이 필요할 때만 아래 브랜치 이름을 변경하세요.
현재 브랜치 이름은 `main`입니다.

In [ ]:
####################################################################
# Edit this value to choose the DX-All Suite Git branch.
# Current default branch name is "main"
####################################################################
DX_ALL_SUITE_BRANCH = "main"
####################################################################

save_tutorial_config(git_branch=DX_ALL_SUITE_BRANCH)
paths = setup_tutorial(quiet=True)

print(f"DX-All Suite Git branch: {DX_ALL_SUITE_BRANCH}")
print(f"Updated configuration:    {paths.config_path}")


### 1.2 DX-All Suite 클론

다음 셀은 설정한 위치에 **DX-All Suite가 없을 때만** 아래 명령을 실행합니다. 디렉터리와 브랜치는 위에서 저장한 설정에서 가져오며, 셀에는 `{DX_ALL_SUITE_DIR}`와 `{DX_ALL_SUITE_BRANCH}`로 표시됩니다.

```bash
git clone --depth 1 --shallow-submodules --recurse-submodules --progress \
    --branch main https://github.com/DEEPX-AI/dx-all-suite.git ~/dx-all-suite
```

- `--depth 1 --shallow-submodules`는 모든 저장소의 최신 커밋만 가져옵니다. 수 GB 대신 약 600 MB를 다운로드하고 디스크에 1.6 GB를 사용합니다.
- `--recurse-submodules`는 서브모듈(`dx-runtime`, `dx-compiler`, `dx-modelzoo` 및 그 안의 저장소)도 함께 체크아웃합니다.
- `--progress`는 출력이 터미널이 아닐 때도 진행 상황을 출력하므로 셀 안에서 진행 상황을 볼 수 있습니다.

네트워크에 따라 1~10분 정도 걸립니다. ■ 버튼으로 셀을 중지했다가 나중에 다시 실행할 수 있습니다. 완료되면 마지막 줄에 `Submodule path ... checked out`이 표시됩니다.

이전 클론이 최상위 체크아웃 이후에 중단되었다면 디렉터리에 `.git`은 있지만 일부 서브모듈이 비어 있습니다. 이 경우 셀은 다시 클론하는 대신 다음 명령을 실행합니다.

```bash
git -C ~/dx-all-suite submodule update --init --recursive --depth 1 --progress
```

셀 대신 터미널(**File > New > Terminal**)에서 두 명령 중 하나를 직접 실행해도 됩니다. 아래 상태 셀은 두 경우 모두 동일하게 동작합니다.

In [ ]:
# Clone DX-All Suite only when it is missing. The git commands below are the ones to use in a terminal.
if paths.check("sdk")[0].ok:
    print(f"DX-All Suite is already present: {DX_ALL_SUITE_DIR}")
    print("Nothing to clone. Continue with the next cell.")

elif (DX_ALL_SUITE_DIR / ".git").exists():
    print(f"Finishing an interrupted clone: initializing submodules in {DX_ALL_SUITE_DIR}\n")
    !git -C "{DX_ALL_SUITE_DIR}" submodule update --init --recursive --depth 1 --progress

else:
    print(f"Cloning branch {DX_ALL_SUITE_BRANCH} into {DX_ALL_SUITE_DIR}\n")
    !git clone --depth 1 --shallow-submodules --recurse-submodules --progress \
        --branch "{DX_ALL_SUITE_BRANCH}" https://github.com/DEEPX-AI/dx-all-suite.git "{DX_ALL_SUITE_DIR}"

paths = setup_tutorial(quiet=True)
if not paths.check("sdk")[0].ok:
    raise RuntimeError(
        f"DX-All Suite is still not usable at {DX_ALL_SUITE_DIR}. "
        "Read the git message above (network, disk space, or a non-empty directory), fix it, and run this cell again."
    )
print("\nDX-All Suite is ready:", DX_ALL_SUITE_DIR)


### 1.3 현재 상태와 다음 단계

다음 셀은 지금까지 설치된 항목을 보여 주고, `MISSING`으로 표시된 항목에 대해서는 그것을 제공하는 정확한 단계를 알려 줍니다. 이 노트북으로 돌아올 때마다 다시 실행하세요. 반복 실행해도 안전합니다.

In [ ]:
# Current state of the installation, with the next step for anything that is missing.
paths = setup_tutorial(needs=["sdk_branch", "sdk_submodules", "dx_com", "dx_rt"])

print()
print(f"Jupyter Python     : {sys.executable}")
print(f"DX-Compiler Python : {DX_COMPILER_VENV / 'bin' / 'python'}")
print()

missing = paths.missing
if any(result.name == "sdk" for result in missing):
    missing = [result for result in missing if result.name == "sdk"]

if not missing:
    print("Section 1 is complete. Continue with section 2 (DX-Compiler) and section 3 (DX-Runtime).")
else:
    print("Next steps, in order:")
    for number, result in enumerate(missing, 1):
        print(f"  {number}. {result.name}: {result.hint}")


### 1.4 DX-All Suite 구성

클론에는 세 개의 Git 서브모듈이 들어 있습니다.
 - `dx-compiler` (**DX-Compiler**): ONNX 모델을 DXNN으로 변환합니다. 2절에서 설치합니다.
 - `dx-runtime` (**DX-Runtime**): 컴파일된 모델을 DEEPX NPU에서 실행합니다. 3절에서 설치합니다.
 - `dx-modelzoo`: 이후 튜토리얼에서 사용하는 사전 컴파일된 모델과 컴파일러 설정입니다.

In [ ]:
!tree -L 1 "{DX_ALL_SUITE_DIR}"


DX-Runtime은 다음을 제공합니다.
 - `DX-APP`: 사용자 애플리케이션 관점에서 DX NPU 사용법을 보여 주는 DEEPX 애플리케이션 템플릿
 - `DX-FW`: NPU 펌웨어 바이너리
 - `DX-RT`: DX NPU를 사용한 추론 작업을 최적화해 실행하도록 설계된 프레임워크
 - `DX-NPU Driver`: DX NPU용 Linux 커널 드라이버
 - `DX-STREAM`: DX NPU용 GStreamer 기반 비전 AI 애플리케이션 개발 도구

In [ ]:
!tree -L 1 "{DX_RUNTIME_DIR}"


## 2. DX-Compiler 설치

자세한 내용은 [DX-All Suite 설치 가이드](https://github.com/DEEPX-AI/dx-all-suite/blob/main/docs/source/02_Setting_Up_Environment.md)를 참고하세요.

DX-Compiler 환경은 사전 빌드된 바이너리를 제공하며 소스 코드는 포함하지 않습니다. 설치 스크립트는 각 모듈을 원격 서버에서 다운로드합니다.

| 명령 | 설치 대상 | `sudo` |
|---|---|---|
| `./dx-compiler/install.sh --target=dx_com` | 전용 Python 환경에 설치되는 DX-COM 컴파일러(`dxcom`) | 이미 설치되어 있어도 `apt-get install python3-dev python3-venv`를 실행 |
| `./dx-compiler/install.sh --target=dx_tron` | DX-TRON 모델 뷰어(`.deb` 패키지) | `apt-get`으로 패키지 설치 |
| `./dx-compiler/install.sh` | 둘 다 | 위 두 가지 모두 |

이 튜토리얼은 2.1에서 DX-COM을 설치합니다. DX-TRON은 선택 사항이며 2.3에서 다룹니다.

### 2.1 DX-COM 설치

실행할 명령은 다음과 같습니다.

```bash
cd ~/dx-all-suite
./dx-compiler/install.sh --target=dx_com
```

`run-jupyter-lab.sh`는 튜토리얼의 Python 환경을 활성화하지 않은 채 JupyterLab을 시작하므로, 이 명령은 노트북 셀에서 실행하든 터미널에서 실행하든 시스템 `python3`를 사용합니다. 설치 스크립트는 `dx-compiler/venv-dx-compiler-local`에 자체 환경을 만들며 Jupyter 환경은 건드리지 않습니다.

*어디서* 실행할지를 결정하는 유일한 요소는 `sudo`입니다. 설치 스크립트는 `python3-dev`와 `python3-venv`를 위해 항상 `sudo apt-get`을 호출하는데, 노트북 셀은 비밀번호 프롬프트에 답할 수 없습니다. 그래서 아래 세 셀은 다음을 수행합니다.

1. **확인**: `dxcom`이 이미 설치되어 있는지, 이 환경에서 `sudo`가 비밀번호 없이 동작하는지 확인합니다.
2. **설치**: 위 명령을 실행합니다(확인 셀이 실행해도 된다고 알려 줄 때만 실행하세요).
3. **검증**: `dxcom`이 존재하는지 확인합니다.

다운로드 용량은 수백 MB이며 2~10분 정도 걸립니다. 출력 끝에 녹색 `[HINT] dx_com installation completed!` 블록이 표시되며, 참고용으로 아래에 실어 두었습니다.

In [ ]:
# 1. Check: is DX-COM installed, and can this notebook run the installer?
if DXCOM_PATH.is_file():
    print(f"DX-COM is already installed: {DXCOM_PATH}")
    print("The install cell will skip itself; continue with the verify cell.")
elif subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    print("Passwordless sudo is available. Run the next cell to install DX-COM here.")
else:
    print("sudo needs a password on this host, so the installer cannot run in a cell.")
    print("Open File > New > Terminal, run the two commands below, then run the verify cell:\n")
    print(f'cd {shlex.quote(str(DX_ALL_SUITE_DIR))}')
    print("./dx-compiler/install.sh --target=dx_com")


In [ ]:
# 2. Install DX-COM (same command as in a terminal). Skipped when dxcom is already installed.
if DXCOM_PATH.is_file():
    print("DX-COM is already installed, skipping:", DXCOM_PATH)
else:
    !cd "{DX_ALL_SUITE_DIR}" && ./dx-compiler/install.sh --target=dx_com


설치가 정상적으로 끝나면 설치 스크립트가 아래 힌트 블록을 출력합니다. 생성된 가상 환경 `dx-compiler/venv-dx-compiler-local`은 Jupyter 환경과 분리되어 있습니다. 2.2의 셀들은 각 `!` 줄의 셸 안에서 `source venv-dx-compiler-local/bin/activate`로 이 환경을 활성화하므로 Jupyter 환경은 바뀌지 않습니다.

<pre style="color: green">
[HINT] ==================================================================== 
[HINT]   dx_com installation completed! 
[HINT]  
[HINT]   To use dx_com, activate the virtual environment first: 
[HINT]     $ source <path_to_dx_all_suite>/dx-compiler/venv-dx-compiler-local/bin/activate 
[HINT]  
[HINT]   Then you can run dxcom: 
[HINT]     $ dxcom -h 
[HINT]  
[HINT] ==================================================================== 
</pre>

다음으로 DX-Compiler가 설치한 구성 요소를 확인합니다.

In [ ]:
# 3. Verify: stop here with the fix if dxcom is still missing.
paths = setup_tutorial(needs=["dx_com"], quiet=True)
paths.require("dx_com")
print("DX-COM is ready:", DXCOM_PATH)


In [ ]:
!tree -L 2 "{DX_COMPILER_DIR}"

이제 dx_com 아래의 파일과 폴더를 살펴보겠습니다.

In [ ]:
!tree -L 1 "{DX_COMPILER_DIR}"/dx_com

# There are three ONNX files as sample models
!tree -L 2 "{DX_COMPILER_DIR}"/dx_com/sample_models

### 2.2 DX-Compiler 검증

2.1의 검증 셀에서 `dxcom`이 존재하는 것은 이미 확인했습니다. 이 절에서는 실제로 실행해 봅니다. 먼저 도움말을 출력하고, 그다음 샘플 모델을 실제로 컴파일합니다.

In [ ]:
print(f"DX-COM workspace: {DX_COM_DIR}")
print(f"dxcom executable: {DXCOM_PATH}")


#### 2.2.1 `dxcom` 도움말 확인

`dxcom`은 설치 스크립트가 만든 가상 환경 안에 있으므로, 터미널에서는 호출하기 전에 그 환경을 활성화해야 합니다. 다음 셀은 정확히 아래 명령을 실행합니다.

```bash
cd ~/dx-all-suite/dx-compiler
source venv-dx-compiler-local/bin/activate
dxcom -h
```

노트북의 `!` 줄은 매번 새 셸을 시작하므로 세 명령을 `&&`로 한 줄에 이어 붙였습니다. 활성화는 그 셸에만 적용되며 Jupyter 환경은 바뀌지 않습니다.

In [ ]:
!cd "{DX_COMPILER_DIR}" && source venv-dx-compiler-local/bin/activate && dxcom -h


#### 2.2.2 `MobileNetV2-1.onnx`를 컴파일해 `MobileNetV2-1.dxnn` 생성

다음 셀은 아래 명령을 실행합니다. 컴파일러는 각 단계를 출력하고, `--gen_log` 옵션은 같은 내용을 `compiler.log`에도 기록합니다. 몇 초에서 1분 정도 걸립니다.

```bash
cd ~/dx-all-suite/dx-compiler
source venv-dx-compiler-local/bin/activate
cd dx_com
dxcom -m sample_models/onnx/MobileNetV2-1.onnx \
      -c sample_models/json/MobileNetV2-1.json \
      -o output/MobileNetV2-1 \
      --gen_log
```

In [ ]:
!cd "{DX_COMPILER_DIR}" && source venv-dx-compiler-local/bin/activate && cd dx_com && \
    dxcom -m sample_models/onnx/MobileNetV2-1.onnx \
          -c sample_models/json/MobileNetV2-1.json \
          -o output/MobileNetV2-1 \
          --gen_log


`MobileNetV2-1.dxnn` 파일이 생성되었는지 확인합니다.

In [ ]:
!tree -h "{DX_COM_DIR}/sample_models" "{DX_COM_DIR}/output"


`--gen_log` 옵션은 컴파일 로그를 `compiler.log` 파일에 기록합니다.

저장된 로그 메시지를 확인해 보겠습니다.

In [ ]:
!cat "{DX_COM_DIR}"/output/MobileNetV2-1/compiler.log

### 2.3 DX-Tron

**DX-TRON**은 DEEPX 툴체인으로 컴파일한 `.dxnn` 모델 파일을 살펴보는 그래픽 시각화 도구입니다. 

모델 구조를 불러와 확인하고, 색으로 구분된 그래프로 NPU와 CPU 사이의 워크로드 분배를 볼 수 있습니다. 

DX-TRON을 사용하면 모델 실행 흐름을 더 잘 이해하고 전체 성능을 개선할 수 있습니다.

> **참고:** DX-TRON은 SDK와 함께 계속 배포되지만 더 이상 적극적으로 유지보수되지 않습니다. 컴파일된 모델의 리포트로는 `dxcom --export_html`이 만드는 HTML 요약(튜토리얼 05)을 권장합니다. DX-TRON은 `.dxnn` 파일을 빠르게 시각적으로 살펴볼 때 여전히 유용합니다.

<img src="assets/sc-dxtron.png" style="max-width: 600px;">

**주요 기능:**
- **.dxnn 파일 지원**: DEEPX 툴체인으로 컴파일한 모델 파일을 불러와 시각화합니다.
- **워크로드 시각화**: 워크로드 실행을 색으로 구분해 표시합니다.
- 빨간색: NPU에서 실행되는 연산
- 파란색: CPU 또는 호스트에서 실행되는 연산
- **모델 탐색 컨트롤**: 왼쪽 아래의 뒤로 가기 화살표로 언제든 모델 개요 화면으로 돌아갈 수 있습니다.
- **대화형 노드 검사**: 그래프의 노드를 더블 클릭하면 해당 연산의 상세 정보를 볼 수 있습니다.
- 참고: DX-TRON은 DXNN을 지원하기 위해 [netron](https://netron.app/)을 기반으로 개발되었습니다.

#### 2.3.1 DX-TRON 설치

DX-TRON은 Debian 패키지로 배포되므로 설치 스크립트가 `sudo apt-get`을 사용합니다.

```bash
cd ~/dx-all-suite
./dx-compiler/install.sh --target=dx_tron
```

2.1과 마찬가지로 세 셀이 이어집니다. **확인**(`dxtron`이 설치되어 있는지, `sudo`가 비밀번호 없이 동작하는지), **설치**(위 명령. 확인 셀이 실행해도 된다고 알려 줄 때만 실행하거나 터미널에서 실행하세요), **검증**입니다.

In [ ]:
# 1. Check: is DX-TRON installed, and can this notebook run the installer?
if shutil.which("dxtron"):
    print(f"DX-TRON is already installed: {shutil.which('dxtron')}")
    print("The install cell will skip itself; continue with the verify cell.")
elif subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    print("Passwordless sudo is available. Run the next cell to install DX-TRON here.")
else:
    print("sudo needs a password on this host, so the installer cannot run in a cell.")
    print("Open File > New > Terminal, run the two commands below, then run the verify cell:\n")
    print(f'cd {shlex.quote(str(DX_ALL_SUITE_DIR))}')
    print("./dx-compiler/install.sh --target=dx_tron")


In [ ]:
# 2. Install DX-TRON (same command as in a terminal). Skipped when dxtron is already installed.
if shutil.which("dxtron"):
    print("DX-TRON is already installed, skipping:", shutil.which("dxtron"))
else:
    !cd "{DX_ALL_SUITE_DIR}" && ./dx-compiler/install.sh --target=dx_tron


In [ ]:
# 3. Verify: stop here with the fix if dxtron is still missing.
paths = setup_tutorial(needs=["dx_tron"], quiet=True)
paths.require("dx_tron")
print("DX-TRON is ready:", shutil.which("dxtron"))


#### 2.3.2 DX-TRON 실행

다음 셀은 2.2.2에서 컴파일한 MobileNetV2 모델을 DX-TRON에서 엽니다. 창이 열려 있는 동안 셀은 실행 중 상태로 유지됩니다.
> **참고:** 위의 중지 버튼('■')을 클릭해 `dxtron`을 종료할 수 있습니다!

In [ ]:
!dxtron "{DX_COM_DIR}"/output/MobileNetV2-1/MobileNetV2-1.dxnn

## 3. DX-Runtime 설치
자세한 내용은 [DX-All Suite 설치 가이드](https://github.com/DEEPX-AI/dx-all-suite/blob/main/docs/source/02_Setting_Up_Environment.md)를 참고하세요.

### 3.1 (선택) 설치 전 사전 요구 사항 (`Orangepi-5 plus`인 경우만)
 - Orange Pi 공식 이미지를 사용하는 경우 커널 헤더가 설치되어 있지 않을 수 있습니다. NPU 드라이버를 설치하려면 커널 헤더가 필요합니다.
 - [여기](../../docs/orangepi5p.md)에 링크된 문서를 참고하세요.

### 3.2 (선택) 설치 전 사전 요구 사항 (`Raspberrypi-5`인 경우만)
 - PCIe는 기본적으로 Gen2로 설정되어 있습니다. 대역폭을 높이려면 Gen3으로 설정할 수 있습니다. 
 - [여기](../../docs/raspberrypi5.md)에 링크된 문서를 참고하세요. 

DX-Runtime 환경에는 각 모듈의 소스 코드가 포함되어 있습니다. 저장소는 `./dx-runtime` 아래에 Git 서브모듈(`dx_rt_npu_linux_driver`, `dx_fw`, `dx_rt`, `dx_app`, `dx_stream`)로 관리됩니다.

DX-Runtime 설치 스크립트의 모든 옵션을 살펴보겠습니다.

In [ ]:
!bash "{DX_RUNTIME_DIR}"/install.sh --help

### 3.1 DX-Runtime 설치

이후 튜토리얼에서 DX-APP과 DX-STREAM을 사용하므로 `--all`로 전부 설치합니다.

```bash
cd ~/dx-all-suite
./dx-runtime/install.sh --all
```

핵심 런타임(NPU 드라이버, 펌웨어, DX-RT)만 필요하다면 `--all` 대신 `--runtime-only`를 사용하세요.

설치 스크립트는 전 과정에서 `sudo`를 사용합니다. NPU 커널 드라이버를 빌드하고 로드하며, Debian 패키지를 설치하고, `dxrt.service` 데몬을 등록합니다. 질문은 하지 않습니다. 2.1과 마찬가지로 세 셀이 이어집니다. **확인**(설치되어 있는지, `sudo`가 비밀번호 없이 동작하는지), **설치**(위 명령. 확인 셀이 실행해도 된다고 알려 줄 때만 실행하거나 터미널에서 실행하세요), **검증**입니다. 10~30분 정도 걸리며 대부분 DX-RT, DX-APP, DX-STREAM 빌드 시간입니다.

NPU 드라이버 설치 후에는 **재부팅**이 필요하며, 펌웨어 업데이트 후에는 설치 스크립트가 완전한 전원 종료를 권장합니다. 재부팅 후 `./run-jupyter-lab.sh`를 다시 시작하고, 이 노트북을 열어 첫 번째 코드 셀을 실행한 뒤 3.2부터 계속 진행하세요.

In [ ]:
# 1. Check: is DX-Runtime installed, and can this notebook run the installer?
runtime_cli = shutil.which("dxcli") or shutil.which("dxrt-cli")
if runtime_cli:
    print(f"DX-Runtime is already installed: {runtime_cli}")
    print("The install cell will skip itself; continue with the verify cell.")
elif subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    print("Passwordless sudo is available. Run the next cell to install DX-Runtime here.")
else:
    print("sudo needs a password on this host, so the installer cannot run in a cell.")
    print("Open File > New > Terminal, run the two commands below, reboot, then run the verify cell:\n")
    print(f'cd {shlex.quote(str(DX_ALL_SUITE_DIR))}')
    print("./dx-runtime/install.sh --all")


In [ ]:
# 2. Install DX-Runtime (same command as in a terminal). Skipped when DX-RT is already installed. Reboot afterwards.
if shutil.which("dxcli") or shutil.which("dxrt-cli"):
    print("DX-Runtime is already installed, skipping:", shutil.which("dxcli") or shutil.which("dxrt-cli"))
else:
    !cd "{DX_ALL_SUITE_DIR}" && ./dx-runtime/install.sh --all


In [ ]:
# 3. Verify. The NPU device appears only after the reboot; the kernel driver and service are checked in 3.2.
paths = setup_tutorial(needs=["dx_rt"], quiet=True)
if paths.ok:
    print("DX-Runtime is installed. If you have not rebooted since the driver installation, reboot now, "
          "start ./run-jupyter-lab.sh again, and continue with 3.2.")
else:
    print("DX-Runtime is not installed yet:", paths.missing[0].detail)
    print("Complete the install step above, reboot, and run this cell again.")


### 3.2 설치 검증

설치 디렉터리와 Git 브랜치는 설정 셀에서 이미 `dx-tutorials/config.json`에 저장되었습니다. 이 검증은 저장소, 브랜치, 서브모듈, DX-Compiler 환경, DX-Runtime CLI, NPU 장치 노드를 확인합니다. 그다음 셀들은 PCIe 링크, 커널 드라이버, 서비스를 더 자세히 살펴봅니다.

In [ ]:
# Verify everything that Tutorial 01 installs. Anything MISSING is listed with the step that provides it.
paths = setup_tutorial(needs=["sdk_branch", "sdk_submodules", "dx_com", "dx_rt", "npu"])

print()
if paths.ok:
    print(f"Installation verified. Settings are stored in {CONFIG_PATH}")
else:
    print("Installation is incomplete. Complete the steps listed above and run this cell again.")


In [ ]:
# Check the PCIe generation and lane count of the NPU. Reading LnkCap/LnkSta needs root.
# DX-M1 performs best on PCIe Gen3 (8GT/s) x4; the host decides what is negotiated.
if subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
    !sudo lspci -vvv | grep -iA 33 accelerators | grep -E "LnkCap|LnkSta"
else:
    print("sudo needs a password here. Run this in a terminal instead:")
    print("  sudo lspci -vvv | grep -iA 33 accelerators | grep -E 'LnkCap|LnkSta'")


In [ ]:
# Check NPU kernel driver status -> expected: dxrt_driver, dx_dma
!lsmod | grep dxrt

In [ ]:
# Check NPU device files -> expected: '/dev/dxrt0'
!ls /dev/dxrt*

In [ ]:
# Check NPU daemon status -> expected: dxrt.service - DX-RT Service is active (running) state
!systemctl status dxrt.service

### 3.3 권장: 선택한 모델만 다운로드

다음 코드 셀은 `SELECTED_MODELS`를 허용 목록으로 사용해 해당 모델만 다운로드합니다. 이 튜토리얼에서 권장하는 기본 방식입니다. 다른 모델 집합이 필요하면 셀을 실행하기 전에 목록을 수정하세요.

> **경고 — 꼭 필요한 경우가 아니면 전체 다운로드 명령을 사용하지 마세요.**  
> `!cd $DX_ALL_SUITE_DIR/dx-runtime/dx_app && bash setup.sh <<< ""`를 실행하면 대화형 프롬프트에 빈 답변이 입력됩니다. 빈 답변은 모든 카테고리와 모든 모델을 선택하므로 349개 모델이 전부 다운로드됩니다. 전체 집합은 많은 네트워크 트래픽과 약 29 GB의 저장 공간이 필요합니다.

권장 셀은 `SELECTED_MODELS`를 `setup.sh --models`에 전달하고 `--no-force`를 사용해 이미 있는 모델 파일은 다시 다운로드하지 않습니다. 나열된 24개 모델은 튜토리얼 02~24에서 사용하는 모델이며, 네트워크에 따라 다운로드에 몇 분 정도 걸립니다.

In [ ]:
# Download all models
#!cd "{DX_ALL_SUITE_DIR}"/dx-runtime/dx_app && bash setup.sh <<< ""

In [ ]:
# Download the *.dxnn models used by this tutorial and by the later tutorials

SELECTED_MODELS = shlex.join([
    "sfa3d_608x608",
    "deepmar_resnet50",
    "ResNet50",
    "depth_anything_v2_vits-1",
    "ViT-B-32-256-datacomp_s34b_b86k-1",
    "3ddfa_v2_mobilnet0.5_120x120",
    "SCRFD500M",
    "mediapipe-hand-detector-1",
    "HandLandmarkLite_1",
    "DnCNN_15",
    "zero_dce_pp-1",
    "yolo26s-seg",
    "superpoint-1",
    "yolo26s-obb",
    "yolo26s",
    "YOLOV12N_PPU",
    "yolopv2-1",
    "yolo26s-pose",
    "pidnet-s-1",
    "RealESRGAN_x4-1",
    "YoloV7",
    "YoloV7_PPU",
    "yolov8s_pose",
    "yolov8n_seg",
])

MODEL_MANIFEST_PATH = DX_APP_DIR / "scripts" / "modelzoo_manifest.json"

# Check the downloadable model list
#!cat "{MODEL_MANIFEST_PATH}" | grep name

# Download selected models only
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force

In [ ]:
# Let's check what models were downloaded in the workspace directory below.
!tree "{DX_WORKSPACE_DIR}"/res/models

# Let's check what videos were downloaded in the workspace directory below.
!tree "{DX_WORKSPACE_DIR}"/res/videos

### 3.4 DX-RT가 제공하는 유용한 도구

DX-RT가 지원하는 CLI 도구를 살펴보겠습니다.

In [ ]:
!ls -l /usr/local/bin | grep -E "dx|model"

**현재 CLI 이름과 하위 호환 이름**

DX-RT는 현재 CLI 바이너리를 `/usr/local/bin`에 설치합니다. 이전 명령 이름은 현재 바이너리를 가리키는 심볼릭 링크로 계속 사용할 수 있습니다.

| 현재 이름 | 이전 호환 이름 | 구현 |
|---|---|---|
| `dxcli` | `dxrt-cli` | `dxrt-cli -> dxcli` |
| `dxrun` | `run_model` | `run_model -> dxrun` |
| `dxparse` | `parse_model` | `parse_model -> dxparse` |

따라서 두 이름 모두 같은 바이너리를 실행하고 같은 옵션을 받습니다. 도움말에는 호출에 사용한 이름이 표시될 수 있습니다. `dxbenchmark`, `dxtop`, `dxrtd`는 기존 이름을 유지하며 독립 바이너리로 설치됩니다.

#### 3.4.1 dxbenchmark

`dxbenchmark`는 컴파일된 .dxnn 모델을 실행해 기능을 테스트하고 성능을 측정하는 CLI 도구입니다.

In [ ]:
# Check dxbenchmark tool
!dxbenchmark -h

In [ ]:
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"
WORKSPACE_DIR.mkdir(exist_ok=True)   # dxbenchmark writes its reports into the current directory

# Benchmark every downloaded model (20 models x 5 loops: expect a few minutes).
!cd "{WORKSPACE_DIR}" && dxbenchmark --dir "{MODEL_DIR}" -l 5


In [ ]:
# Three report formats are generated: csv, json, and html.
!ls "{WORKSPACE_DIR}"/DXBENCHMARK*


웹 브라우저를 열어 `dxbenchmark` 결과를 HTML 형식으로 확인합니다.

In [ ]:
import base64
from pathlib import Path

from IPython.display import Javascript, display

report_directories = {
    WORKSPACE_DIR.resolve(),
    Path.cwd().resolve(),
}
benchmark_reports = [
    report
    for directory in report_directories
    for report in directory.glob("DXBENCHMARK_*.html")
]
if not benchmark_reports:
    raise FileNotFoundError("No DXBENCHMARK_*.html report was found. Run dxbenchmark first.")

latest_report = max(benchmark_reports, key=lambda path: path.stat().st_mtime)
encoded_report = base64.b64encode(latest_report.read_bytes()).decode("ascii")

display(Javascript(f'''(() => {{
    const binary = atob("{encoded_report}");
    const bytes = Uint8Array.from(binary, character => character.charCodeAt(0));
    const blob = new Blob([bytes], {{type: "text/html"}});
    const reportUrl = URL.createObjectURL(blob);
    window.open(reportUrl, "_blank");
    setTimeout(() => URL.revokeObjectURL(reportUrl), 60000);
}})();'''))
print(f"Opened benchmark report: {latest_report}")
print("If no new tab appeared, your browser blocked the popup. Open the file above directly in a browser.")

#### 3.4.2 dxtop

`dxtop`은 사용률, 온도, 메모리 같은 DEEPX NPU 지표를 실시간으로 모니터링하는 htop 스타일의 도구입니다.

> dxtop의 출력 형식은 Jupyter Notebook 코드 셀에서 올바르게 표시되지 않습니다. 대신 **별도의 터미널을 열어** dxtop 명령을 실행하세요. </br>
> 별도의 터미널을 여는 방법: `File > New > Terminal`
> 
> ![](assets/open-terminal.png)

> <img src="assets/sc-dxtop.png" style="max-width: 600px;">

In [ ]:
# Check dxtop tool - Open a separate terminal and run 'dxtop'
!dxtop -h
#!dxtop

#### 3.4.3 dxcli

`dxcli`(호환성을 위해 `dxrt-cli`로도 사용 가능)는 DEEPX DX-RT 장치를 조회·모니터링하고 NPU 펌웨어를 관리합니다.

In [ ]:
# Check dxcli tool
!dxcli -h

In [ ]:
# Check the NPU status
!dxcli -s

In [ ]:
# Check the NPU F/W version
!dxcli -i | grep FW

#### 선택: NPU 펌웨어 플래시

DX-Runtime 설치 스크립트가 이 SDK 버전에 맞는 펌웨어를 이미 기록하므로 이 단계는 보통 **필요하지 않습니다**. DEEPX 지원팀이 재플래시를 요청하거나 다른 SDK 브랜치로 전환한 뒤에만 사용하세요. 다음 셀에서 `FLASH_FIRMWARE = True`로 설정하면 활성화되며, `False`이면 셀은 수행할 작업만 출력합니다.

명령은 다음과 같습니다.

```bash
sudo systemctl stop dxrt.service
dxcli -u ~/dx-all-suite/dx-runtime/dx_fw/m1/latest/mdot2/fw.bin   # DX-H1은 h1/fw.bin
sleep 5
sudo systemctl start dxrt.service
```

In [ ]:
####################################################################
FLASH_FIRMWARE = False   # set to True only when a re-flash is really required
####################################################################

FIRMWARE_PATH = DX_RUNTIME_DIR / "dx_fw" / "m1" / "latest" / "mdot2" / "fw.bin"
# For DX-H1 hardware use: DX_RUNTIME_DIR / "dx_fw" / "m1" / "latest" / "h1" / "fw.bin"

if not FLASH_FIRMWARE:
    print(f"Skipped. Firmware image that would be written: {FIRMWARE_PATH}")
elif not FIRMWARE_PATH.is_file():
    raise FileNotFoundError(f"Firmware image was not found: {FIRMWARE_PATH}")
else:
    !sudo systemctl stop dxrt.service
    !dxcli -u "{FIRMWARE_PATH}"
    !sleep 5
    !sudo systemctl start dxrt.service


In [ ]:
# Check the NPU F/W version
!dxcli -i | grep FW

#### 3.4.4 dxrun

`dxrun`(호환성을 위해 `run_model`로도 사용 가능)은 컴파일된 .dxnn 모델을 실행해 기능을 테스트하고 성능을 측정하는 CLI 도구입니다.

In [ ]:
# Check dxrun tool
!dxrun -h

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"
!dxrun -m "{MODEL_FILE}"

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

# Open the model in DX-TRON. The cell stays busy while the window is open; stop it with the ■ button.
!dxtron "{MODEL_FILE}"

`yolo26-s_640x640.dxnn`은 두 부분(NPU와 CPU)으로 구성됩니다.

 ![](assets/sc-yolo26s.png)

NPU가 지원하지 않는 연산자는 CPU 오프로딩을 통해 ONNX Runtime으로 CPU에서 처리됩니다. 

`dxrun`에 `--use-ort` 옵션을 추가하면 NPU와 CPU 부분을 모두 실행하므로 .dxnn AI 파이프라인 전체의 성능을 측정할 수 있습니다.

`--use-ort` 옵션을 생략하면 NPU 부분만 성능을 측정합니다.

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

!echo "============================= NO ORT ================================="
!dxrun -m "{MODEL_FILE}" -l 200 | grep FPS
!echo "======================================================================"
!echo ""
!echo "============================= USE ORT ================================"
!dxrun -m "{MODEL_FILE}" -l 200 --use-ort | grep FPS
!echo "======================================================================"

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

# -t 5: For 5 seconds, -v: Shows NPU Processing Time and Latency
!dxrun -m "{MODEL_FILE}" --use-ort -t 5 -v

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"

# --profiler writes profiler.json into the current directory, so run it inside the tutorial workspace.
WORKSPACE_DIR.mkdir(exist_ok=True)
!cd "{WORKSPACE_DIR}" && dxrun -m "{MODEL_FILE}" --use-ort -t 5 -v --profiler


#### 3.4.5 dxparse

`dxparse`(호환성을 위해 `parse_model`로도 사용 가능)는 컴파일된 .dxnn 모델을 읽어 구조, 입출력, 메타데이터를 표시하는 명령줄 도구입니다.

In [ ]:
!dxparse -h

In [ ]:
MODEL_FILE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"
!dxparse -m "{MODEL_FILE}" -v

## 4. 문제 해결

| 증상 | 표시되는 내용 | 원인 | 해결 |
|---|---|---|---|
| 설치 셀이 `sudo`에서 멈춤 | `sudo: a terminal is required to read the password` | 비밀번호 없는 sudo가 설정되지 않아 설치 스크립트가 셀 안에서 비밀번호를 물어볼 수 없음 | 출력된 명령을 터미널(File > New > Terminal)에서 실행한 뒤 검증 셀을 실행 |
| 클론 셀 실패 | `fatal: destination path ... already exists and is not an empty directory` | 대상 위치에 git 체크아웃이 아닌 파일이 있음 | 디렉터리를 옮기거나 1.1에서 다른 위치를 선택한 뒤 셀을 다시 실행 |
| `dxcom`이 계속 MISSING | 상태 표에 `[MISSING] dx_com` | DX-COM 설치 스크립트가 끝까지 실행되지 않음 | 위로 스크롤해 설치 출력을 확인. `[HINT]` 블록이 있어야 함. 2.1을 다시 실행 |
| NPU 장치 없음 | `[MISSING] npu` 또는 `ls: cannot access '/dev/dxrt*'` | 커널 드라이버가 로드되지 않음. 보통 3.1 이후 호스트를 재부팅하지 않은 경우 | 재부팅. 그다음 `lsmod \| grep dxrt`를 확인하고, 장치가 여전히 없으면 호스트 전원을 완전히 껐다가 다시 켬 |
| `dxrt.service`가 실행되지 않음 | `systemctl status dxrt.service`에 `Active: failed` | 드라이버 또는 펌웨어 불일치 | `sudo systemctl restart dxrt.service`. 다시 실패하면 터미널에서 `./dx-runtime/install.sh --runtime-only` 실행 |
| 모델이 다운로드되지 않음 | `setup.sh`가 알 수 없는 모델 이름을 보고 | 이름이 Model Zoo 매니페스트와 다름(대소문자 구분) | `dx_app`에서 `grep name scripts/modelzoo_manifest.json`으로 정확한 이름 확인 |

## 5. 요약

| 항목 | DX-Compiler | DX-Runtime |
|---|---|---|
| 역할 | 모델 컴파일 | 모델 추론 실행 |
| 입력 | ONNX | DXNN |
| 출력 | `.dxnn` | 추론 결과 |
| 시스템 | x86_64 전용 | x86_64, aarch64 |
| 설치 명령 | `./dx-compiler/install.sh --target=dx_com` | `./dx-runtime/install.sh --all` |
| 주요 CLI | `dxcom` (`venv-dx-compiler-local` 안) | `dxcli`, `dxrun`, `dxparse`, `dxbenchmark`, `dxtop` |

### 5.1 완료 체크리스트

- [ ] `config.json`이 DX-All Suite 디렉터리를 가리키고 1.3의 상태 셀에서 모든 항목이 `OK`로 표시됨
- [ ] `dxcom -h`가 실행되고 `MobileNetV2-1.dxnn`이 컴파일됨 (2.2)
- [ ] `dxtron`으로 `.dxnn` 파일을 열 수 있음 (2.3, 선택)
- [ ] `/dev/dxrt0`이 존재하고 `dxrt.service`가 활성 상태이며 `dxcli -s`에 NPU가 표시됨 (3.2)
- [ ] 선택한 모델이 공유 워크스페이스에 다운로드됨 (3.3)
- [ ] `dxrun`과 `dxbenchmark`로 NPU에서 모델을 실행함 (3.4)

> **다음:** 튜토리얼 02로 이동해 여기서 다운로드한 모델로 DX-APP 예제(분류, 검출, 포즈, 세그멘테이션)를 실행해 보세요.